# FastAPI 라우터(Router) 개념 실습

`contact_app/routers/contacts.py`를 열어보기 전에, **라우터가 왜 필요하고 어떻게 동작하는지**를 작은 장난감 예제("책 관리 API")로 직접 실행해보면서 눈으로 확인하는 노트북입니다.

- 실제 서버(uvicorn)를 띄우지 않고, `TestClient`로 파이썬 프로세스 안에서 바로 요청을 날려봅니다. (`tests/test_api.py`에서 쓰는 것과 같은 도구입니다.)
- 실제 프로젝트의 `contacts.py`, `categories.py`와 **똑같은 패턴** (`APIRouter(prefix=..., tags=...)`)을 그대로 사용합니다.
- 모든 셀은 이미 한 번 실행된 결과가 저장되어 있어서 그냥 읽기만 해도 되고, `Run All`로 직접 다시 실행해봐도 똑같은 결과가 나옵니다.

In [1]:
import warnings
warnings.filterwarnings("ignore")  # 이 노트북 안에서만 경고 메시지 숨김

from fastapi import FastAPI, APIRouter, Depends, HTTPException, status
from fastapi.testclient import TestClient
from pydantic import BaseModel
from typing import Optional

## 실험 1 — 라우터 없이, `app`에 직접 엔드포인트 등록하기

가장 단순한 방식입니다. `@app.get(...)`, `@app.post(...)`처럼 앱 객체에 바로 함수를 붙입니다.

`app.openapi()["paths"]`는 "지금 이 앱에 등록된 모든 URL 목록"을 보여줍니다. 사실 이게 바로 `/docs` 페이지(Swagger UI)가 자동으로 그려주는 정보의 원본이기도 해요.

In [2]:
app_no_router = FastAPI()


@app_no_router.get("/books")
def list_books_v1():
    return {"books": []}


@app_no_router.post("/books")
def create_book_v1():
    return {"created": True}


for path, methods in app_no_router.openapi()["paths"].items():
    print(path, list(methods.keys()))

/books ['get', 'post']


지금은 함수 2개뿐이라 괜찮아 보이지만, 실제 프로젝트처럼 `/contacts`, `/categories`, `/auth` 관련 함수가 30~50개가 되면 어떻게 될까요?

- `main.py` 파일 하나가 몇백 줄짜리 파일이 되어버립니다.
- "연락처 관련 코드만 보고 싶다" 가 안 됩니다 — 전부 한 파일에 섞여있으니까요.
- URL 앞에 매번 `/contacts`를 손으로 반복해서 써야 합니다 (오타 위험, 나중에 경로를 바꾸려면 수십 군데를 고쳐야 함).

**라우터(`APIRouter`)** 는 이 문제를 "같은 주제의 엔드포인트를 하나의 파일/객체로 묶고, 공통 접두어(prefix)를 한 번에 지정"해서 해결합니다.

**비유**: `app`이 건물 전체라면, `APIRouter`는 "1층 전체를 담당하는 설계도 뭉치"예요. 그 설계도 뭉치를 통째로 건물에 끼워 넣는 것이 `include_router()`입니다.

## 실험 2 — `APIRouter`로 분리하기 (실제 `contacts.py`와 동일한 패턴)

실제 코드 `contact_app/routers/contacts.py`는 이렇게 시작합니다:

```python
router = APIRouter(prefix="/contacts", tags=["contacts"])

@router.post("", ...)
def create(...): ...

@router.get("", ...)
def read(...): ...
```

여기서는 "연락처" 대신 "책"으로 **똑같은 구조**를 만들어봅니다. (`Depends(get_current_user)`, `Depends(get_db)` 자리는 각각 가짜 로그인 함수 / 가짜 DB 리스트로 대체했습니다.)

In [3]:
router = APIRouter(prefix="/books", tags=["books"])

_fake_db: list[dict] = []
_next_id = 1


class BookCreate(BaseModel):
    title: str
    author: str


class BookUpdate(BaseModel):
    title: Optional[str] = None
    author: Optional[str] = None


class BookResponse(BaseModel):
    id: int
    title: str
    author: str


def get_current_user():
    return {"username": "demo_user"}


@router.post("", response_model=BookResponse, status_code=status.HTTP_201_CREATED)
def create_book(data: BookCreate, current_user: dict = Depends(get_current_user)):
    global _next_id
    book = {"id": _next_id, "title": data.title, "author": data.author}
    _fake_db.append(book)
    _next_id += 1
    return book


@router.get("", response_model=list[BookResponse])
def list_books(current_user: dict = Depends(get_current_user)):
    return _fake_db


@router.patch("/{book_id}", response_model=BookResponse)
def update_book(book_id: int, data: BookUpdate, current_user: dict = Depends(get_current_user)):
    for book in _fake_db:
        if book["id"] == book_id:
            if data.title is not None:
                book["title"] = data.title
            if data.author is not None:
                book["author"] = data.author
            return book
    raise HTTPException(status_code=404, detail="책을 찾을 수 없습니다.")


@router.delete("/{book_id}", status_code=status.HTTP_204_NO_CONTENT)
def delete_book(book_id: int, current_user: dict = Depends(get_current_user)):
    for book in _fake_db:
        if book["id"] == book_id:
            _fake_db.remove(book)
            return
    raise HTTPException(status_code=404, detail="책을 찾을 수 없습니다.")

### `router`를 `app`에 연결하기 *전에도* prefix가 이미 붙어있다

`@router.get("")`이라고만 썼는데, `router.routes`를 확인해보면 이미 `/books`가 앞에 붙어 있는 걸 볼 수 있습니다. `prefix="/books"`가 이 라우터에 등록되는 모든 경로에 자동으로 적용되기 때문입니다.

In [4]:
for r in router.routes:
    print(r.methods, r.path)

{'POST'} /books
{'GET'} /books
{'PATCH'} /books/{book_id}
{'DELETE'} /books/{book_id}


### 이제 진짜 `app`에 끼워 넣기 (`include_router`)

`main.py`의 `app.include_router(contacts_router)`와 완전히 같은 코드입니다. 연결 후 `app.openapi()["paths"]`를 보면, 라우터 안의 모든 경로가 그대로 `app`에 등록된 것을 확인할 수 있습니다.

In [5]:
app = FastAPI(title="책 관리 데모")
app.include_router(router)

for path, methods in app.openapi()["paths"].items():
    print(path, list(methods.keys()))

/books ['get', 'post']
/books/{book_id} ['patch', 'delete']


## 실제로 요청 보내보기 (`TestClient`)

`TestClient`는 서버를 진짜로 띄우지 않고도, 마치 브라우저가 요청을 보낸 것처럼 파이썬 안에서 바로 테스트할 수 있게 해주는 도구입니다.

아래 셀들을 순서대로 실행하면서, "책 등록 → 조회 → 수정 → 삭제"까지 매 단계마다 `_fake_db`(가짜 데이터베이스 역할을 하는 파이썬 리스트)의 상태가 어떻게 바뀌는지 하나씩 따라가 봅니다.

In [6]:
client = TestClient(app)

print("1) 책 등록 전 목록 조회: GET /books")
res = client.get("/books")
print(res.status_code, res.json())

1) 책 등록 전 목록 조회: GET /books
200 []


In [7]:
print("2) 책 등록: POST /books")
res = client.post("/books", json={"title": "이상한 나라의 앨리스", "author": "루이스 캐럴"})
print(res.status_code, res.json())

2) 책 등록: POST /books
201 {'id': 1, 'title': '이상한 나라의 앨리스', 'author': '루이스 캐럴'}


In [8]:
print("3) 책 등록 후 목록 조회: GET /books")
res = client.get("/books")
print(res.status_code, res.json())

3) 책 등록 후 목록 조회: GET /books
200 [{'id': 1, 'title': '이상한 나라의 앨리스', 'author': '루이스 캐럴'}]


In [9]:
print("4) 책 정보 수정: PATCH /books/1")
res = client.patch("/books/1", json={"title": "이상한 나라의 앨리스 (개정판)"})
print(res.status_code, res.json())

4) 책 정보 수정: PATCH /books/1
200 {'id': 1, 'title': '이상한 나라의 앨리스 (개정판)', 'author': '루이스 캐럴'}


In [10]:
print("5) 책 삭제: DELETE /books/1")
res = client.delete("/books/1")
print(res.status_code, repr(res.text))

5) 책 삭제: DELETE /books/1
204 ''


In [11]:
print("6) 삭제 후 목록 조회: GET /books")
res = client.get("/books")
print(res.status_code, res.json())

6) 삭제 후 목록 조회: GET /books
200 []


In [12]:
print("7) 없는 책 수정 시도: PATCH /books/999")
res = client.patch("/books/999", json={"title": "없는 책"})
print(res.status_code, res.json())

7) 없는 책 수정 시도: PATCH /books/999
404 {'detail': '책을 찾을 수 없습니다.'}


## 이 장난감 예제 ↔ 실제 프로젝트 대응표

| 이 노트북 | 실제 프로젝트 (`contact_app/routers/contacts.py`) |
|---|---|
| `router = APIRouter(prefix="/books", tags=["books"])` | `router = APIRouter(prefix="/contacts", tags=["contacts"])` |
| `_fake_db` (파이썬 리스트) | 실제 Postgres 데이터베이스 (SQLAlchemy 세션을 통해 `crud.py`가 접근) |
| `get_current_user()`가 그냥 고정값을 반환 | `Depends(get_current_user)`가 실제 로그인 세션/쿠키를 검사 (`routers/auth.py`) |
| `app.include_router(router)` 1개 | `main.py`에서 `auth_router`, `contacts_router`, `categories_router` 3개를 동일한 패턴으로 연결 |
| `TestClient`로 파이썬 안에서 직접 호출 | 실제로는 브라우저의 `fetch()`가 `http://127.0.0.1:포트/contacts`로 진짜 네트워크 요청 |

**핵심 요약**: 라우터는 "같은 주제(연락처, 카테고리, 로그인)의 URL들을 하나의 파일에 몰아넣고, 공통 접두어(prefix)와 공통 태그(tags — `/docs` 문서에서 그룹으로 묶여 보이게 함)를 한 번만 지정하는 도구"입니다. `main.py`는 이 라우터 뭉치들을 조립만 할 뿐, 실제 로직(입력 검증, DB 조회, 에러 처리)은 각 라우터 파일 안에 있습니다.

다음 단계로 `contact_app/routers/contacts.py`를 열어서, 이 노트북의 `_fake_db` 자리에 실제로 무엇이 들어가는지(`Depends(get_db)`, `crud.py`, 실제 SQLAlchemy 모델)를 하나씩 비교해보면 이해가 훨씬 쉬워집니다.